In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm
import os, gc
from scipy.stats import norm
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [3]:
names = locals()

In [4]:
start, end = 1931, 2022
span = end - start + 1
economy = 'CHN'
n_sim = 10_000
seed = 42
std = 0.05
out_dir = RESULTS / 'material-flows' / economy / 'historical'
out_dir.mkdir(parents=True, exist_ok=True)

In [5]:
def make_constant_values(df, n_sim):
    values = np.array(df).astype(np.float32)  # shape (years,)
    return np.tile(values, (n_sim, 1))

In [6]:
def make_uncertain_values(df, std, span, run, seed):
    values = np.array([df]) if type(df) == float else np.array(df)
    if len(values) == 92:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))
    elif len(values) == 1:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))

    return gg.astype(np.float32) 

In [7]:
def make_stock_array(array_2D):
    n, s = array_2D.shape
    out = np.empty((n, s, s), dtype=np.float32)
    out[:] = array_2D.astype(np.float32, copy=False)[:, :, None]
    j = np.arange(s)
    out[:, j[:, None] > j[None, :]] = np.nan   # j > k 处置 NaN
    return out

In [8]:
def cal_dem_rate(lft_2D):
    run, span = lft_2D.shape
    j = np.arange(span, dtype=np.float32)
    invalid = j[:, None] > j[None, :]                     # (span, span)

    lft_3D = np.empty((run, span, span), dtype=np.float32)
    lft_3D[:] = lft_2D.astype(np.float32, copy=False)[:, :, None]
    lft_3D[:, invalid] = np.nan

    age_2d = (j[None, :] - j[:, None]).astype(np.float32) # k - j
    age_3D = np.broadcast_to(age_2d, (run, span, span)).copy()
    age_3D[:, invalid] = np.nan

    ccc = norm.cdf(age_3D, loc=lft_3D, scale=lft_3D * 0.2).astype(np.float32)
    sur_rate_3D = (1.0 - ccc).astype(np.float32)

    dem_rate_3D = np.full_like(ccc, np.nan)
    dem_rate_3D[:, :, 1:] = ccc[:, :, 1:] - ccc[:, :, :-1]
    d = np.arange(span)
    dem_rate_3D[:, d, d] = 0.0
    return sur_rate_3D, dem_rate_3D

In [9]:
region_country = pd.read_excel(DATA / 'contents.xlsx', sheet_name='region_division')
region_country.loc[28, 'region'] = 'NA'
region_country.loc[171, 'region'] = 'NA'

In [10]:
df_cement_production = pd.read_csv(DATA / 'production-142.csv', index_col=0)
df_cement_export = pd.read_csv(RESULTS / 'trade-data' / 'cement-export-142.csv', index_col=0)
df_cement_import = pd.read_csv(RESULTS / 'trade-data' / 'cement-import-142.csv', index_col=0)
df_clinker_export = pd.read_csv(RESULTS / 'trade-data' / 'clinker-export-142.csv', index_col=0)
df_clinker_import = pd.read_csv(RESULTS / 'trade-data' / 'clinker-import-142.csv', index_col=0)
df_cementPro_export = pd.read_csv(RESULTS / 'trade-data' / 'cement-product-export-142.csv', index_col=0)
df_cementPro_import = pd.read_csv(RESULTS / 'trade-data' / 'cement-product-import-142.csv', index_col=0)
df_limestone_export = pd.read_csv(RESULTS / 'trade-data' / 'limestone-export-142.csv', index_col=0)
df_limestone_import = pd.read_csv(RESULTS / 'trade-data' / 'limestone-import-142.csv', index_col=0)
df_gravel_export = pd.read_csv(RESULTS / 'trade-data' / 'gravel-crushed-stone-export-142.csv', index_col=0)
df_gravel_import = pd.read_csv(RESULTS / 'trade-data' / 'gravel-crushed-stone-import-142.csv', index_col=0)
df_sand_export = pd.read_csv(RESULTS / 'trade-data' / 'other-sand-export-142.csv', index_col=0)
df_sand_import = pd.read_csv(RESULTS / 'trade-data' / 'other-sand-import-142.csv', index_col=0)

In [11]:
df_concrete_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='concrete_ratio', index_col=0)
df_C15_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C15_ratio', index_col=0)
df_C16_23_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C16_23_ratio', index_col=0)
df_C24_35_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C24_35_ratio', index_col=0)
df_C35_ratio = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='C35_ratio', index_col=0)
df_cement_content_Pro = pd.read_excel(DATA / 'params-1.xlsx', sheet_name='cement_in_Pro', index_col=0)

In [12]:
# load GNR params 
GNR_CCR = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='CCR', index_col=0)
GNR_ratio_gypsum = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_gypsum', index_col=0)
GNR_ratio_limestone = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_limestone', index_col=0)
GNR_ratio_pozzolana = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_pozzolana', index_col=0)
GNR_ratio_slag = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_slag', index_col=0)
GNR_ratio_flyash = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_flyash', index_col=0)
GNR_ratio_others = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='ratio_others', index_col=0)
GNR_binder_slag = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='binder_slag', index_col=0)
GNR_binder_FAP = pd.read_excel(DATA / 'GNR-params.xlsx', sheet_name='binder_FAP', index_col=0)

In [13]:
# sectoral params
economies_142 = df_cement_production.columns.tolist()
df_ratio_RES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_RES', index_col=0)[economies_142]
df_ratio_NRES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_NRES', index_col=0)[economies_142]
df_ratio_CIV = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='ratio_CIV', index_col=0)[economies_142]

df_lft_RES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_RES', index_col=0)[economies_142]
df_lft_NRES = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_NRES', index_col=0)[economies_142]
df_lft_CIV = pd.read_excel(DATA / 'sector-params.xlsx', sheet_name='lft_CIV', index_col=0)[economies_142]

In [14]:
df_other_params = pd.read_excel(DATA / 'other-params.xlsx', index_col=0)

In [16]:
for var in [
    'cement_production',
    'cement_export',
    'cement_import',
    'clinker_export',
    'clinker_import',
    'cementPro_export',
    'cementPro_import',
    'limestone_export',
    'limestone_import',
    'gravel_export',
    'gravel_import',
    'sand_export',
    'sand_import',
]:
    generated_data = make_constant_values(names[f'df_{var}'].loc[start:end, economy].values, n_sim)
    np.savez_compressed(out_dir / f'{var}.npz', data=generated_data.astype(np.float32))

In [17]:
np.random.seed(seed)
CCR_mode = np.where(GNR_CCR.loc[start:end, economy].values > 0.95, 0.95, GNR_CCR.loc[start:end, economy]).astype(np.float32)
CCR = np.random.triangular(np.ones(span)*0.5, CCR_mode, np.ones(span)*0.95, size=(n_sim, span)).astype(np.float32)
np.savez_compressed(out_dir / 'CCR.npz', data=CCR.astype(np.float32))

In [18]:
for var in [
    'ratio_gypsum',
    'ratio_limestone',
    'ratio_pozzolana',
    'ratio_slag',
    'ratio_flyash',
    'ratio_others',
    'binder_slag',
    'binder_FAP'
]:
    generated_data = make_uncertain_values(names[f'GNR_{var}'].loc[start:end, economy], std, span, n_sim, seed)
    np.savez_compressed(out_dir / f'{var}.npz', data=generated_data.astype(np.float32))

In [19]:
# make uncertain value array for the ratio of concrete, mortar, C15, C16-23, C24-35, C35. 
concrete_ratio = make_uncertain_values(df_concrete_ratio.loc[start:end, economy].values, std, span, n_sim, seed)
mortar_ratio = 100 - concrete_ratio

np.savez_compressed(out_dir / 'concrete_ratio.npz', data=concrete_ratio.astype(np.float32))
np.savez_compressed(out_dir / 'mortar_ratio.npz', data=mortar_ratio.astype(np.float32))
# del concrete_ratio, mortar_ratio

In [20]:
r_C15_ = make_uncertain_values(df_C15_ratio.loc[start:end, economy].values, std, span, n_sim, seed)
r_C16_23_ = make_uncertain_values(df_C16_23_ratio.loc[start:end, economy].values, std, span, n_sim, seed)
r_C24_35_ = make_uncertain_values(df_C24_35_ratio.loc[start:end, economy].values, std, span, n_sim, seed)
r_C35_ = make_uncertain_values(df_C35_ratio.loc[start:end, economy].values, std, span, n_sim, seed)

r_C15 = r_C15_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
r_C16_23 = r_C16_23_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
r_C24_35 = r_C24_35_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)
r_C35 = r_C35_ / (r_C15_ + r_C16_23_ + r_C24_35_ + r_C35_)

np.savez_compressed(out_dir / 'r_C15.npz', data=r_C15.astype(np.float32))
np.savez_compressed(out_dir / 'r_C16_23.npz', data=r_C16_23.astype(np.float32))
np.savez_compressed(out_dir / 'r_C24_35.npz', data=r_C24_35.astype(np.float32))
np.savez_compressed(out_dir / 'r_C35.npz', data=r_C35.astype(np.float32)) 
del r_C15_, r_C16_23_, r_C24_35_, r_C35_
# del r_C15, r_C16_23, r_C24_35, r_C35

In [21]:
fine_to_binder_C15 = make_uncertain_values(2.542840011, std, span, n_sim, seed+15)
fine_to_binder_C16_23 = make_uncertain_values(2.855202053, std, span, n_sim, seed+16)
fine_to_binder_C24_35 = make_uncertain_values(2.083908811, std, span, n_sim, seed+17)
fine_to_binder_C35 = make_uncertain_values(1.728617340, std, span, n_sim, seed+18)
water_to_binder_C15 = make_uncertain_values(0.659834197, std, span, n_sim, seed)
water_to_binder_C16_23 = make_uncertain_values(0.619416533, std, span, n_sim, seed)
water_to_binder_C24_35 = make_uncertain_values(0.496178881, std, span, n_sim, seed)
water_to_binder_C35 = make_uncertain_values(0.395957047, std, span, n_sim, seed)
coarse_to_binder_C15 = make_uncertain_values(4.182638806, std, span, n_sim, seed)
coarse_to_binder_C16_23 = make_uncertain_values(3.271740668, std, span, n_sim, seed)
coarse_to_binder_C24_35 = make_uncertain_values(2.584728036, std, span, n_sim, seed)
coarse_to_binder_C35 = make_uncertain_values(2.471397967, std, span, n_sim, seed)
fine_to_binder_mortar = make_uncertain_values(2.720389651, std, span, n_sim, seed)
water_to_binder_mortar = make_uncertain_values(0.506558494, std, span, n_sim, seed)

np.savez_compressed(out_dir / 'fine_to_binder_C15.npz',     data=fine_to_binder_C15.astype(np.float32))
np.savez_compressed(out_dir / 'fine_to_binder_C16_23.npz',  data=fine_to_binder_C16_23.astype(np.float32))
np.savez_compressed(out_dir / 'fine_to_binder_C24_35.npz',  data=fine_to_binder_C24_35.astype(np.float32))
np.savez_compressed(out_dir / 'fine_to_binder_C35.npz',     data=fine_to_binder_C35.astype(np.float32))
np.savez_compressed(out_dir / 'water_to_binder_C15.npz',    data=water_to_binder_C15.astype(np.float32))
np.savez_compressed(out_dir / 'water_to_binder_C16_23.npz', data=water_to_binder_C16_23.astype(np.float32))
np.savez_compressed(out_dir / 'water_to_binder_C24_35.npz', data=water_to_binder_C24_35.astype(np.float32))
np.savez_compressed(out_dir / 'water_to_binder_C35.npz',    data=water_to_binder_C35.astype(np.float32))
np.savez_compressed(out_dir / 'coarse_to_binder_C15.npz',   data=coarse_to_binder_C15.astype(np.float32))
np.savez_compressed(out_dir / 'coarse_to_binder_C16_23.npz', data=coarse_to_binder_C16_23.astype(np.float32))
np.savez_compressed(out_dir / 'coarse_to_binder_C24_35.npz', data=coarse_to_binder_C24_35.astype(np.float32))
np.savez_compressed(out_dir / 'coarse_to_binder_C35.npz',   data=coarse_to_binder_C35.astype(np.float32))
np.savez_compressed(out_dir / 'fine_to_binder_mortar.npz',  data=fine_to_binder_mortar.astype(np.float32))
np.savez_compressed(out_dir / 'water_to_binder_mortar.npz', data=water_to_binder_mortar.astype(np.float32))

# del fine_to_binder_C15, fine_to_binder_C16_23, fine_to_binder_C24_35, fine_to_binder_C35
# del water_to_binder_C15, water_to_binder_C16_23, water_to_binder_C24_35, water_to_binder_C35
# del coarse_to_binder_C15, coarse_to_binder_C16_23, coarse_to_binder_C24_35, coarse_to_binder_C35
# del fine_to_binder_mortar, water_to_binder_mortar

In [22]:
aa = region_country[region_country['ISO-3'] == economy]['cement_content_in_products'].values.astype(np.float32)
cement_content = make_uncertain_values(aa, std, span, n_sim, seed)
np.savez_compressed(out_dir / 'cement_content.npz', data=cement_content.astype(np.float32))
# del cement_content

In [23]:
r_RES_ = make_uncertain_values(df_ratio_RES.loc[start:end, economy].values, std, span, n_sim, seed)
r_NRES_ = make_uncertain_values(df_ratio_NRES.loc[start:end, economy].values, std, span, n_sim, seed)
r_CIV_ = make_uncertain_values(df_ratio_CIV.loc[start:end, economy].values, std, span, n_sim, seed)
r_RES = r_RES_ / (r_RES_ + r_NRES_ + r_CIV_)
r_NRES = r_NRES_ / (r_RES_ + r_NRES_ + r_CIV_)
r_CIV = r_CIV_ / (r_RES_ + r_NRES_ + r_CIV_)

np.savez_compressed(out_dir / 'r_RES.npz',    data=r_RES.astype(np.float32))
np.savez_compressed(out_dir / 'r_NRES.npz',   data=r_NRES.astype(np.float32))
np.savez_compressed(out_dir / 'r_CIV.npz',    data=r_CIV.astype(np.float32))

del r_RES_, r_NRES_, r_CIV_
del r_RES, r_NRES, r_CIV

In [24]:
lft_RES = make_uncertain_values(df_lft_RES.loc[start:end, economy].values, std, span, n_sim, seed)
lft_NRES = make_uncertain_values(df_lft_NRES.loc[start:end, economy].values, std, span, n_sim, seed)
lft_CIV = make_uncertain_values(df_lft_CIV.loc[start:end, economy].values, std, span, n_sim, seed)

np.savez_compressed(out_dir / 'lft_RES.npz',  data=lft_RES.astype(np.float32))
np.savez_compressed(out_dir / 'lft_NRES.npz', data=lft_NRES.astype(np.float32))
np.savez_compressed(out_dir / 'lft_CIV.npz',  data=lft_CIV.astype(np.float32))

del lft_RES, lft_NRES, lft_CIV

In [25]:
r_waste = np.random.triangular(0.01, 0.015, 0.03, size=(n_sim, span)).astype(np.float32)
np.savez_compressed(out_dir / 'r_waste.npz',  data=r_waste.astype(np.float32))
# del r_waste

In [26]:
r_Ca = make_uncertain_values(1.45402, std, span, n_sim, seed)   # calcareous ratio 
r_Al = make_uncertain_values(0.09921, std, span, n_sim, seed)    # aluminous ratio 
r_Fe = make_uncertain_values(0.01798, std, span, n_sim, seed)   # ferrous ratio 
r_Si = make_uncertain_values(0.09350, std, span, n_sim, seed) # siliceous ratio

np.savez_compressed(out_dir / 'r_Ca.npz',     data=r_Ca.astype(np.float32))
np.savez_compressed(out_dir / 'r_Al.npz',     data=r_Al.astype(np.float32))
np.savez_compressed(out_dir / 'r_Fe.npz',     data=r_Fe.astype(np.float32))
np.savez_compressed(out_dir / 'r_Si.npz',     data=r_Si.astype(np.float32))

del r_Ca, r_Al, r_Fe, r_Si

In [27]:
r_CKD = make_uncertain_values(0.06, std, span, n_sim, seed)
np.savez_compressed(out_dir / 'r_CKD.npz', data=r_CKD.astype(np.float32))
del r_CKD

In [28]:
cement_production = np.load(out_dir / 'cement_production.npz')['data']
cement_export = np.load(out_dir / 'cement_export.npz')['data']
cement_import = np.load(out_dir / 'cement_import.npz')['data']
binder_slag = np.load(out_dir / 'binder_slag.npz')['data']
binder_FAP = np.load(out_dir / 'binder_FAP.npz')['data']
cementPro_export = np.load(out_dir / 'cementPro_export.npz')['data']
cementPro_import = np.load(out_dir / 'cementPro_import.npz')['data']
cement_content = np.load(out_dir / 'cement_content.npz')['data']

In [29]:
#********************    CALCULATION     *************************
cement_consumption_ = cement_production - cement_export + cement_import
cement_consumption = np.where(cement_consumption_ >= 0, cement_consumption_, 0)
cement_sub_slag = cement_consumption * binder_slag
cement_sub_FAP = cement_consumption * binder_FAP
all_binder = cement_consumption + cement_sub_slag + cement_sub_FAP  # cement_consumption * (1 + binder_slag + binder_FAP)

binder_in_exportPro = cementPro_export * cement_content
binder_in_importPro = cementPro_import * cement_content

gggg = binder_in_importPro - binder_in_exportPro
binder_for_concrete = (all_binder * concrete_ratio / 100) + gggg
binder_for_mortar = all_binder * mortar_ratio / 100

concrete_waste = (binder_for_concrete * r_waste).astype(np.float32)
mortar_waste = (binder_for_mortar * r_waste).astype(np.float32)

In [30]:
for name in [
    'cement_consumption', 
    'cement_sub_slag', 
    'cement_sub_FAP', 
    'all_binder',
    'binder_in_exportPro', 
    'binder_in_importPro', 
    'binder_for_concrete', 
    'binder_for_mortar',
    'concrete_waste',
    'mortar_waste'
]:
    np.savez_compressed(out_dir / f'{name}.npz', data=globals()[name].astype(np.float32))
    # del globals()[name]

In [31]:
strengths = ['C15', 'C16_23', 'C24_35', 'C35']
apps = ['RES', 'NRES', 'CIV']


for app in apps:  
    # 1. create create value array for survival and demolition rates 
    lft_app = np.load(out_dir / f'lft_{app}.npz')['data']
    r_app = np.load(out_dir / f'r_{app}.npz')['data']

    sur_rate_app, dem_rate_app = cal_dem_rate(lft_app)

    # 2. allocate inflow for different application 
    binder_for_concrete_app = binder_for_concrete * (1 - r_waste) * r_app
    binder_for_mortar_app = binder_for_mortar * (1 - r_waste) * r_app

    np.savez_compressed(out_dir / f'binder_for_concrete_{app}.npz', data=binder_for_concrete_app.astype(np.float32))
    np.savez_compressed(out_dir / f'binder_for_mortar_{app}.npz', data=binder_for_mortar_app.astype(np.float32))

    # 3. calculate outflow and stock 
    initial_array = make_stock_array(binder_for_concrete_app)
    S_binder_for_concrete_app = initial_array * sur_rate_app
    O_binder_for_concrete_app = initial_array * dem_rate_app

    # np.savez_compressed(out_dir / f'S_binder_for_concrete_{app}.npz', data=S_binder_for_concrete)
    # np.savez_compressed(out_dir / f'O_binder_for_concrete_{app}.npz', data=O_binder_for_concrete)
    
    a_shape = S_binder_for_concrete_app.shape
    S_fine_for_concrete_app   = np.zeros(shape=a_shape)
    S_coarse_for_concrete_app = np.zeros(shape=a_shape)
    S_water_for_concrete_app  = np.zeros(shape=a_shape)
    O_fine_for_concrete_app   = np.zeros(shape=a_shape)
    O_coarse_for_concrete_app = np.zeros(shape=a_shape)
    O_water_for_concrete_app  = np.zeros(shape=a_shape)
    S_concrete_app            = np.zeros(shape=a_shape)
    O_concrete_app            = np.zeros(shape=a_shape)

    for strength in tqdm(strengths):
        r_s   = np.load(out_dir / f'r_{strength}.npz')['data'][:, :, None]
        f2b_s = np.load(out_dir / f'fine_to_binder_{strength}.npz')['data'][:, :, None]
        c2b_s = np.load(out_dir / f'coarse_to_binder_{strength}.npz')['data'][:, :, None]
        w2b_s = np.load(out_dir / f'water_to_binder_{strength}.npz')['data'][:, :, None]

        S_binder_for_concrete_app_strength = S_binder_for_concrete_app          * r_s
        S_fine_for_concrete_app_strength   = S_binder_for_concrete_app_strength * f2b_s
        S_coarse_for_concrete_app_strength = S_binder_for_concrete_app_strength * c2b_s
        S_water_for_concrete_app_strength  = S_binder_for_concrete_app_strength * w2b_s
        S_concrete_app_strength = np.sum([S_binder_for_concrete_app_strength, S_fine_for_concrete_app_strength, S_coarse_for_concrete_app_strength, S_water_for_concrete_app_strength], axis=0)

        O_binder_for_concrete_app_strength = O_binder_for_concrete_app          * r_s
        O_fine_for_concrete_app_strength   = O_binder_for_concrete_app_strength * f2b_s
        O_coarse_for_concrete_app_strength = O_binder_for_concrete_app_strength * c2b_s
        O_water_for_concrete_app_strength  = O_binder_for_concrete_app_strength * w2b_s
        O_concrete_app_strength = np.sum([O_binder_for_concrete_app_strength, O_fine_for_concrete_app_strength, O_coarse_for_concrete_app_strength, O_water_for_concrete_app_strength], axis=0)

        np.savez_compressed(out_dir / f'fine_for_concrete_{app}_{strength}.npz', data=np.diagonal(S_fine_for_concrete_app_strength.T).astype(np.float32))
        np.savez_compressed(out_dir / f'coarse_for_concrete_{app}_{strength}.npz', data=np.diagonal(S_coarse_for_concrete_app_strength.T).astype(np.float32))

        np.savez_compressed(out_dir / f'S_binder_for_concrete_{app}_{strength}_annual.npz', data=np.nansum(S_binder_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'S_fine_for_concrete_{app}_{strength}_annual.npz',   data=np.nansum(S_fine_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'S_coarse_for_concrete_{app}_{strength}_annual.npz', data=np.nansum(S_coarse_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'S_water_for_concrete_{app}_{strength}_annual.npz',  data=np.nansum(S_water_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'S_concrete_{app}_{strength}_annual.npz',            data=np.nansum(S_concrete_app_strength, axis=1).astype(np.float32))

        np.savez_compressed(out_dir / f'O_binder_for_concrete_{app}_{strength}_annual.npz', data=np.nansum(O_binder_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'O_fine_for_concrete_{app}_{strength}_annual.npz',   data=np.nansum(O_fine_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'O_coarse_for_concrete_{app}_{strength}_annual.npz', data=np.nansum(O_coarse_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'O_water_for_concrete_{app}_{strength}_annual.npz',  data=np.nansum(O_water_for_concrete_app_strength, axis=1).astype(np.float32))
        np.savez_compressed(out_dir / f'O_concrete_{app}_{strength}_annual.npz',            data=np.nansum(O_concrete_app_strength, axis=1).astype(np.float32))
        
        S_fine_for_concrete_app   += S_fine_for_concrete_app_strength
        S_coarse_for_concrete_app += S_coarse_for_concrete_app_strength
        S_water_for_concrete_app  += S_water_for_concrete_app_strength
        
        O_binder_for_concrete_app += O_binder_for_concrete_app_strength
        O_fine_for_concrete_app   += O_fine_for_concrete_app_strength
        O_coarse_for_concrete_app += O_coarse_for_concrete_app_strength
        O_water_for_concrete_app  += O_water_for_concrete_app_strength
        
        S_concrete_app            += S_concrete_app_strength
        O_concrete_app            += O_concrete_app_strength

        S_binder_for_concrete_app_strength_cohort = S_binder_for_concrete_app_strength[:, :, -1]
        np.savez_compressed(out_dir / f'S_binder_for_concrete_{app}_{strength}_cohort.npz', data=S_binder_for_concrete_app_strength_cohort.astype(np.float32))


    np.savez_compressed(out_dir / f'concrete_{app}.npz', data=np.diagonal(S_concrete_app.T).astype(np.float32))
    np.savez_compressed(out_dir / f'fine_for_concrete_{app}.npz',  data=np.diagonal(S_fine_for_concrete_app.T).astype(np.float32))
    np.savez_compressed(out_dir / f'coarse_{app}.npz',  data=np.diagonal(S_coarse_for_concrete_app.T).astype(np.float32))
    np.savez_compressed(out_dir / f'water_for_concrete_{app}.npz',  data=np.diagonal(S_water_for_concrete_app.T).astype(np.float32))
    
    np.savez_compressed(out_dir / f'S_binder_for_concrete_{app}_annual.npz', data=np.nansum(S_binder_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'S_fine_for_concrete_{app}_annual.npz',   data=np.nansum(S_fine_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'S_coarse_for_concrete_{app}_annual.npz', data=np.nansum(S_coarse_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'S_water_for_concrete_{app}_annual.npz',  data=np.nansum(S_water_for_concrete_app, axis=1).astype(np.float32))

    np.savez_compressed(out_dir / f'O_binder_for_concrete_{app}_annual.npz', data=np.nansum(O_binder_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_fine_for_concrete_{app}_annual.npz',   data=np.nansum(O_fine_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_coarse_for_concrete_{app}_annual.npz', data=np.nansum(O_coarse_for_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_water_for_concrete_{app}_annual.npz',  data=np.nansum(O_water_for_concrete_app, axis=1).astype(np.float32))

    np.savez_compressed(out_dir / f'S_concrete_{app}_annual.npz',            data=np.nansum(S_concrete_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_concrete_{app}_annual.npz',            data=np.nansum(O_concrete_app, axis=1).astype(np.float32))

    S_binder_for_concrete_app_cohort = S_binder_for_concrete_app[:, :, -1]
    np.savez_compressed(out_dir / f'S_binder_for_concrete_{app}_cohort.npz', data=S_binder_for_concrete_app_cohort.astype(np.float32))


    initial_array2 = make_stock_array(binder_for_mortar_app)
    S_binder_for_mortar_app = initial_array2 * sur_rate_app
    O_binder_for_mortar_app = initial_array2 * dem_rate_app

    S_fine_for_mortar_app  = S_binder_for_mortar_app * fine_to_binder_mortar[:, :, None]
    S_water_for_mortar_app = S_binder_for_mortar_app * water_to_binder_mortar[:, :, None]

    O_fine_for_mortar_app  = O_binder_for_mortar_app * fine_to_binder_mortar[:, :, None]
    O_water_for_mortar_app = O_binder_for_mortar_app * water_to_binder_mortar[:, :, None]

    S_mortar_app = np.sum([S_binder_for_mortar_app, S_fine_for_mortar_app, S_water_for_mortar_app], axis=0)
    O_mortar_app = np.sum([O_binder_for_mortar_app, O_fine_for_mortar_app, O_water_for_mortar_app], axis=0)

    np.savez_compressed(out_dir / f'mortar_{app}.npz', data=np.diagonal(S_mortar_app.T).astype(np.float32))
    np.savez_compressed(out_dir / f'fine_for_mortar_{app}.npz',  data=np.diagonal(S_fine_for_mortar_app.T).astype(np.float32))
    np.savez_compressed(out_dir / f'water_for_mortar_{app}.npz', data=np.diagonal(S_water_for_mortar_app.T).astype(np.float32))

    np.savez_compressed(out_dir / f'S_binder_for_mortar_{app}_annual.npz', data=np.nansum(S_binder_for_mortar_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'S_fine_for_mortar_{app}_annual.npz',   data=np.nansum(S_fine_for_mortar_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'S_water_for_mortar_{app}_annual.npz',  data=np.nansum(S_water_for_mortar_app, axis=1).astype(np.float32))
    
    np.savez_compressed(out_dir / f'O_binder_for_mortar_{app}_annual.npz', data=np.nansum(O_binder_for_mortar_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_fine_for_mortar_{app}_annual.npz',   data=np.nansum(O_fine_for_mortar_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_water_for_mortar_{app}_annual.npz',  data=np.nansum(O_water_for_mortar_app, axis=1).astype(np.float32))

    np.savez_compressed(out_dir / f'S_mortar_{app}_annual.npz', data=np.nansum(S_mortar_app, axis=1).astype(np.float32))
    np.savez_compressed(out_dir / f'O_mortar_{app}_annual.npz', data=np.nansum(O_mortar_app, axis=1).astype(np.float32))

    S_binder_for_mortar_app_cohort = S_binder_for_mortar_app[:, :, -1]
    np.savez_compressed(out_dir / f'S_binder_for_mortar_{app}_cohort.npz', data=S_binder_for_mortar_app_cohort.astype(np.float32))


100%|██████████| 4/4 [00:16<00:00,  4.21s/it]


In [32]:
for strength in strengths:
    fine_for_concrete_RES_strength  = np.load(out_dir / f'fine_for_concrete_RES_{strength}.npz')['data']
    fine_for_concrete_NRES_strength = np.load(out_dir / f'fine_for_concrete_NRES_{strength}.npz')['data']
    fine_for_concrete_CIV_strength  = np.load(out_dir / f'fine_for_concrete_CIV_{strength}.npz')['data']
    fine_for_concrete_strength      = fine_for_concrete_RES_strength + fine_for_concrete_NRES_strength + fine_for_concrete_CIV_strength
    np.savez_compressed(out_dir / f'fine_for_concrete_{strength}.npz', data=fine_for_concrete_strength.astype(np.float32))

In [33]:
S_binder_for_concrete_annual = np.zeros(shape=(n_sim, span))
S_fine_for_concrete_annual   = np.zeros(shape=(n_sim, span))
S_coarse_for_concrete_annual = np.zeros(shape=(n_sim, span))
S_water_for_concrete_annual  = np.zeros(shape=(n_sim, span))
S_concrete_annual            = np.zeros(shape=(n_sim, span))

S_binder_for_mortar_annual   = np.zeros(shape=(n_sim, span))
S_fine_for_mortar_annual     = np.zeros(shape=(n_sim, span))
S_water_for_mortar_annual    = np.zeros(shape=(n_sim, span))
S_mortar_annual              = np.zeros(shape=(n_sim, span))

O_binder_for_concrete_annual = np.zeros(shape=(n_sim, span))
O_fine_for_concrete_annual   = np.zeros(shape=(n_sim, span))
O_coarse_for_concrete_annual = np.zeros(shape=(n_sim, span))
O_water_for_concrete_annual  = np.zeros(shape=(n_sim, span))
O_concrete_annual            = np.zeros(shape=(n_sim, span))

O_binder_for_mortar_annual   = np.zeros(shape=(n_sim, span))
O_fine_for_mortar_annual     = np.zeros(shape=(n_sim, span))
O_water_for_mortar_annual    = np.zeros(shape=(n_sim, span))
O_mortar_annual              = np.zeros(shape=(n_sim, span))

for app in apps:
    S_binder_for_concrete_app_annual = np.load(out_dir / f'S_binder_for_concrete_{app}_annual.npz', )['data']
    S_fine_for_concrete_app_annual   = np.load(out_dir / f'S_fine_for_concrete_{app}_annual.npz', )['data']
    S_coarse_for_concrete_app_annual = np.load(out_dir / f'S_coarse_for_concrete_{app}_annual.npz', )['data']
    S_water_for_concrete_app_annual  = np.load(out_dir / f'S_water_for_concrete_{app}_annual.npz', )['data']
    S_concrete_app_annual            = np.load(out_dir / f'S_concrete_{app}_annual.npz', )['data']

    S_binder_for_concrete_annual += S_binder_for_concrete_app_annual
    S_fine_for_concrete_annual   += S_fine_for_concrete_app_annual
    S_coarse_for_concrete_annual += S_coarse_for_concrete_app_annual
    S_water_for_concrete_annual  += S_water_for_concrete_app_annual
    S_concrete_annual            += S_concrete_app_annual

    S_binder_for_mortar_app_annual = np.load(out_dir / f'S_binder_for_mortar_{app}_annual.npz', )['data']
    S_fine_for_mortar_app_annual   = np.load(out_dir / f'S_fine_for_mortar_{app}_annual.npz', )['data']
    S_water_for_mortar_app_annual  = np.load(out_dir / f'S_water_for_mortar_{app}_annual.npz', )['data']
    S_mortar_app_annual            = np.load(out_dir / f'S_mortar_{app}_annual.npz', )['data']

    S_binder_for_mortar_annual += S_binder_for_mortar_app_annual
    S_fine_for_mortar_annual   += S_fine_for_mortar_app_annual
    S_water_for_mortar_annual  += S_water_for_mortar_app_annual
    S_mortar_annual            += S_mortar_app_annual

    O_binder_for_concrete_app_annual = np.load(out_dir / f'O_binder_for_concrete_{app}_annual.npz', )['data']
    O_fine_for_concrete_app_annual   = np.load(out_dir / f'O_fine_for_concrete_{app}_annual.npz', )['data']
    O_coarse_for_concrete_app_annual = np.load(out_dir / f'O_coarse_for_concrete_{app}_annual.npz', )['data']
    O_water_for_concrete_app_annual  = np.load(out_dir / f'O_water_for_concrete_{app}_annual.npz', )['data']
    O_concrete_app_annual            = np.load(out_dir / f'O_concrete_{app}_annual.npz', )['data']

    O_binder_for_concrete_annual += O_binder_for_concrete_app_annual
    O_fine_for_concrete_annual   += O_fine_for_concrete_app_annual
    O_coarse_for_concrete_annual += O_coarse_for_concrete_app_annual
    O_water_for_concrete_annual  += O_water_for_concrete_app_annual
    O_concrete_annual            += O_concrete_app_annual

    O_binder_for_mortar_app_annual = np.load(out_dir / f'O_binder_for_mortar_{app}_annual.npz', )['data']
    O_fine_for_mortar_app_annual   = np.load(out_dir / f'O_fine_for_mortar_{app}_annual.npz', )['data']
    O_water_for_mortar_app_annual  = np.load(out_dir / f'O_water_for_mortar_{app}_annual.npz', )['data']
    O_mortar_app_annual            = np.load(out_dir / f'O_mortar_{app}_annual.npz', )['data']

    O_binder_for_mortar_annual += O_binder_for_mortar_app_annual
    O_fine_for_mortar_annual   += O_fine_for_mortar_app_annual
    O_water_for_mortar_annual  += O_water_for_mortar_app_annual
    O_mortar_annual            += O_mortar_app_annual


In [34]:
np.savez_compressed(out_dir / 'S_binder_for_concrete_annual.npz', data=S_binder_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_fine_for_concrete_annual.npz',   data=S_fine_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_coarse_for_concrete_annual.npz', data=S_coarse_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_water_for_concrete_annual.npz',  data=S_water_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_concrete_annual.npz',            data=S_concrete_annual.astype(np.float32))

np.savez_compressed(out_dir / 'S_binder_for_mortar_annual.npz', data=S_binder_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_fine_for_mortar_annual.npz',   data=S_fine_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_water_for_mortar_annual.npz',  data=S_water_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_mortar_annual.npz',            data=S_mortar_annual.astype(np.float32))

np.savez_compressed(out_dir / 'O_binder_for_concrete_annual.npz', data=O_binder_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_fine_for_concrete_annual.npz',   data=O_fine_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_coarse_for_concrete_annual.npz', data=O_coarse_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_water_for_concrete_annual.npz',  data=O_water_for_concrete_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_concrete_annual.npz',            data=O_concrete_annual.astype(np.float32))

np.savez_compressed(out_dir / 'O_binder_for_mortar_annual.npz', data=O_binder_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_fine_for_mortar_annual.npz',   data=O_fine_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_water_for_mortar_annual.npz',  data=O_water_for_mortar_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_mortar_annual.npz',            data=O_mortar_annual.astype(np.float32))

In [35]:
S_binder_RES_annual = np.zeros(shape=(n_sim, span))
S_binder_NRES_annual = np.zeros(shape=(n_sim, span))
S_binder_CIV_annual = np.zeros(shape=(n_sim, span))
S_fine_RES_annual = np.zeros(shape=(n_sim, span))
S_fine_NRES_annual = np.zeros(shape=(n_sim, span))
S_fine_CIV_annual = np.zeros(shape=(n_sim, span))

S_binder_annual = np.zeros(shape=(n_sim, span))
S_fine_annual = np.zeros(shape=(n_sim, span))
S_water_annual = np.zeros(shape=(n_sim, span))

S_RES_annual = np.zeros(shape=(n_sim, span))
S_NRES_annual = np.zeros(shape=(n_sim, span))
S_CIV_annual = np.zeros(shape=(n_sim, span))
S_Total_annual = np.zeros(shape=(n_sim, span))

O_binder_annual = np.zeros(shape=(n_sim, span))
O_fine_annual = np.zeros(shape=(n_sim, span))
O_water_annual = np.zeros(shape=(n_sim, span))

O_RES_annual = np.zeros(shape=(n_sim, span))
O_NRES_annual = np.zeros(shape=(n_sim, span))
O_CIV_annual = np.zeros(shape=(n_sim, span))
O_Total_annual = np.zeros(shape=(n_sim, span))

for material in ['concrete', 'mortar']:
    S_binder_for_material_RES_annual = np.load(out_dir / f'S_binder_for_{material}_RES_annual.npz')['data']
    S_binder_for_material_NRES_annual = np.load(out_dir / f'S_binder_for_{material}_NRES_annual.npz')['data']
    S_binder_for_material_CIV_annual = np.load(out_dir / f'S_binder_for_{material}_CIV_annual.npz')['data']

    S_fine_for_material_RES_annual = np.load(out_dir / f'S_fine_for_{material}_RES_annual.npz')['data']
    S_fine_for_material_NRES_annual = np.load(out_dir / f'S_fine_for_{material}_NRES_annual.npz')['data']
    S_fine_for_material_CIV_annual = np.load(out_dir / f'S_fine_for_{material}_CIV_annual.npz')['data']

    S_binder_for_material_annual = np.load(out_dir / f'S_binder_for_{material}_annual.npz')['data']
    S_fine_for_material_annual = np.load(out_dir / f'S_fine_for_{material}_annual.npz')['data']
    S_water_for_material_annual = np.load(out_dir / f'S_water_for_{material}_annual.npz')['data']

    S_material_RES_annual = np.load(out_dir / f'S_{material}_RES_annual.npz')['data']
    S_material_NRES_annual = np.load(out_dir / f'S_{material}_NRES_annual.npz')['data']
    S_material_CIV_annual = np.load(out_dir / f'S_{material}_CIV_annual.npz')['data']
    S_material_annual = np.load(out_dir / f'S_{material}_annual.npz')['data']

    O_binder_for_material_annual = np.load(out_dir / f'O_binder_for_{material}_annual.npz')['data']
    O_fine_for_material_annual = np.load(out_dir / f'O_fine_for_{material}_annual.npz')['data']
    O_water_for_material_annual = np.load(out_dir / f'O_water_for_{material}_annual.npz')['data']

    O_material_RES_annual = np.load(out_dir / f'O_{material}_RES_annual.npz')['data']
    O_material_NRES_annual = np.load(out_dir / f'O_{material}_NRES_annual.npz')['data']
    O_material_CIV_annual = np.load(out_dir / f'O_{material}_CIV_annual.npz')['data']
    O_material_annual = np.load(out_dir / f'O_{material}_annual.npz')['data']

    S_binder_RES_annual += S_binder_for_material_RES_annual
    S_binder_NRES_annual += S_binder_for_material_NRES_annual
    S_binder_CIV_annual += S_binder_for_material_CIV_annual

    S_fine_RES_annual += S_fine_for_material_RES_annual
    S_fine_NRES_annual += S_fine_for_material_NRES_annual
    S_fine_CIV_annual += S_fine_for_material_CIV_annual

    S_binder_annual += S_binder_for_material_annual
    S_fine_annual += S_fine_for_material_annual
    S_water_annual += S_water_for_material_annual

    S_RES_annual += S_material_RES_annual
    S_NRES_annual += S_material_NRES_annual
    S_CIV_annual += S_material_CIV_annual
    
    S_Total_annual += S_material_annual

    O_binder_annual += O_binder_for_material_annual
    O_fine_annual += O_fine_for_material_annual
    O_water_annual += O_water_for_material_annual

    O_RES_annual += O_material_RES_annual
    O_NRES_annual += O_material_NRES_annual
    O_CIV_annual += O_material_CIV_annual

    O_Total_annual += O_material_annual


S_coarse_RES_annual = np.load(out_dir / 'S_coarse_for_concrete_RES_annual.npz')['data']
S_coarse_NRES_annual = np.load(out_dir / 'S_coarse_for_concrete_NRES_annual.npz')['data']
S_coarse_CIV_annual = np.load(out_dir / 'S_coarse_for_concrete_CIV_annual.npz')['data']

S_coarse_annual = np.load(out_dir / 'S_coarse_for_concrete_annual.npz')['data']
O_coarse_annual = np.load(out_dir / 'O_coarse_for_concrete_annual.npz')['data']

In [36]:
np.savez_compressed(out_dir / 'S_binder_RES_annual.npz', data=S_binder_RES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_binder_NRES_annual.npz', data=S_binder_NRES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_binder_CIV_annual.npz', data=S_binder_CIV_annual.astype(np.float32))

np.savez_compressed(out_dir / 'S_fine_RES_annual.npz', data=S_fine_RES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_fine_NRES_annual.npz', data=S_fine_NRES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_fine_CIV_annual.npz', data=S_fine_CIV_annual.astype(np.float32))

np.savez_compressed(out_dir / 'S_coarse_RES_annual.npz', data=S_coarse_RES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_coarse_NRES_annual.npz', data=S_coarse_NRES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_coarse_CIV_annual.npz', data=S_coarse_CIV_annual.astype(np.float32))

np.savez_compressed(out_dir / 'S_binder_annual.npz', data=S_binder_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_fine_annual.npz', data=S_fine_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_coarse_annual.npz', data=S_coarse_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_water_annual.npz', data=S_water_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_RES_annual.npz', data=S_RES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_NRES_annual.npz', data=S_NRES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_CIV_annual.npz', data=S_CIV_annual.astype(np.float32))
np.savez_compressed(out_dir / 'S_Total_annual.npz', data=S_Total_annual.astype(np.float32))

np.savez_compressed(out_dir / 'O_binder_annual.npz', data=O_binder_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_fine_annual.npz', data=O_fine_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_coarse_annual.npz', data=O_coarse_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_water_annual.npz', data=O_water_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_RES_annual.npz', data=O_RES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_NRES_annual.npz', data=O_NRES_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_CIV_annual.npz', data=O_CIV_annual.astype(np.float32))
np.savez_compressed(out_dir / 'O_Total_annual.npz', data=O_Total_annual.astype(np.float32))

In [37]:
fine_total = np.zeros(shape=(n_sim, span))
coarse_total = np.zeros(shape=(n_sim, span))
water_total = np.zeros(shape=(n_sim, span))

concrete = np.zeros(shape=(n_sim, span))
mortar = np.zeros(shape=(n_sim, span))

for app in apps:
    fine_for_concrete_app = np.load(out_dir / f'fine_for_concrete_{app}.npz')['data']
    fine_for_mortar_app = np.load(out_dir / f'fine_for_mortar_{app}.npz')['data']
    water_for_concrete_app = np.load(out_dir / f'water_for_concrete_{app}.npz')['data']
    water_for_mortar_app = np.load(out_dir / f'water_for_mortar_{app}.npz')['data']

    fine_app = fine_for_concrete_app + fine_for_mortar_app
    water_app = water_for_concrete_app + water_for_mortar_app
    coarse_app = np.load(out_dir / f'coarse_{app}.npz')['data']

    fine_total += fine_app
    coarse_total += coarse_app
    water_total += water_app

    concrete += np.load(out_dir / f'concrete_{app}.npz')['data']
    mortar += np.load(out_dir / f'mortar_{app}.npz')['data']

    np.savez_compressed(out_dir / f'fine_{app}.npz', data=fine_app.astype(np.float32))


In [38]:
np.savez_compressed(out_dir / 'fine_total.npz', data=fine_total.astype(np.float32))
np.savez_compressed(out_dir / 'coarse_total.npz', data=coarse_total.astype(np.float32))
np.savez_compressed(out_dir / 'water_total.npz', data=water_total.astype(np.float32))
np.savez_compressed(out_dir / 'concrete.npz', data=concrete.astype(np.float32))
np.savez_compressed(out_dir / 'mortar.npz', data=mortar.astype(np.float32))

In [39]:
sand_export = np.load(out_dir / 'sand_export.npz')['data']
sand_import = np.load(out_dir / 'sand_import.npz')['data']
gravel_export = np.load(out_dir / 'gravel_export.npz')['data']
gravel_import = np.load(out_dir / 'gravel_import.npz')['data']

sand_production_ = fine_total + sand_export - sand_import
gravel_production_ = coarse_total + gravel_export - gravel_import
excess_sand = np.where(sand_production_ < 0, -sand_production_, 0)
excess_gravel = np.where(gravel_production_ < 0, -gravel_production_, 0)
sand_stock = excess_sand.cumsum(axis=1)
gravel_stock = excess_gravel.cumsum(axis=1)

consumed_sand_stock = np.where((sand_stock > 0) & (sand_production_ > 0), np.where(sand_production_ <= sand_stock, sand_production_, sand_stock), 0)
consumed_gravel_stock = np.where((gravel_stock > 0) & (gravel_production_ > 0), np.where(gravel_production_ <= gravel_stock, gravel_production_, gravel_stock), 0)

sand_stock = sand_stock - consumed_sand_stock
gravel_stock = gravel_stock - consumed_gravel_stock

sand_production = np.where(sand_production_ >= consumed_sand_stock, sand_production_ - consumed_sand_stock, 0)
gravel_production = np.where(gravel_production_ >= consumed_gravel_stock, gravel_production_ - consumed_gravel_stock, 0)

EoLWaste = O_Total_annual

region = region_country[region_country['ISO-3'] == economy]['region'].values[0]
aa = df_other_params.loc['rec_rate', region]
bb = df_other_params.loc['down_rate', region]
rec_rate = make_uncertain_values([aa], std, span, n_sim, seed)
down_rate = make_uncertain_values([bb], std, span, n_sim, seed)

EoLWaste_recycled = EoLWaste * rec_rate
EoLWaste_downcycled = EoLWaste * down_rate 
EoLWaste_disposed = EoLWaste * (1 - rec_rate - down_rate)

aggregate_recycled_ = EoLWaste_recycled 
# aggregate_virgin = fine_total + (coarse_total - aggregate_recycled)

sand_substitution_rate = 0
sand_substitution = sand_production * sand_substitution_rate

aggregate_recycled = np.where(aggregate_recycled_ > gravel_production, gravel_production, aggregate_recycled_)
gravel_extraction = gravel_production - aggregate_recycled 

sand_extraction = sand_production - sand_substitution
aggregate_virgin = gravel_extraction + sand_extraction 

ratio_gypsum = np.load(out_dir / 'ratio_gypsum.npz')['data']
ratio_limestone = np.load(out_dir / 'ratio_limestone.npz')['data']
ratio_pozzolana = np.load(out_dir / 'ratio_pozzolana.npz')['data']
ratio_slag = np.load(out_dir / 'ratio_slag.npz')['data']
ratio_flyash = np.load(out_dir / 'ratio_flyash.npz')['data']
ratio_others = np.load(out_dir / 'ratio_others.npz')['data']

SCM_gypsum = cement_production * ratio_gypsum
SCM_limestone = cement_production * ratio_limestone
SCM_pozzolana = cement_production * ratio_pozzolana
SCM_slag = cement_production * ratio_slag 
SCM_flyash = cement_production * ratio_flyash
SCM_others = cement_production * ratio_others 

clinker_consumption = cement_production * CCR

clinker_export = np.load(out_dir / 'clinker_export.npz')['data']
clinker_import = np.load(out_dir / 'clinker_import.npz')['data']
limestone_export = np.load(out_dir / 'limestone_export.npz')['data']
limestone_import = np.load(out_dir / 'limestone_import.npz')['data']

clinker_production_ = clinker_consumption + clinker_export - clinker_import
excess_clinker = np.where(clinker_production_ < 0, -clinker_production_, 0)
gg = excess_clinker.cumsum(axis=1)
clinker_stock = gg.copy()
clinker_stock[:, 3:] = gg[:, 3:] - gg[:, :-3]
clinker_production = np.where(clinker_production_ >= clinker_stock, clinker_production_ - clinker_stock, 0)

r_CKD = np.load(out_dir / 'r_CKD.npz')['data']
r_Ca = np.load(out_dir / 'r_Ca.npz')['data']
r_Al = np.load(out_dir / 'r_Al.npz')['data']
r_Fe = np.load(out_dir / 'r_Fe.npz')['data']
r_Si = np.load(out_dir / 'r_Si.npz')['data']

CKD = clinker_production * r_CKD  

RawMat_Ca = clinker_production * r_Ca 
RawMat_Al = clinker_production * r_Al 
RawMat_Fe = clinker_production * r_Fe 
RawMat_Si = clinker_production * r_Si 

limestone_ = RawMat_Ca + limestone_export - limestone_import
excess_limestone = np.where(limestone_ < 0, -limestone_, 0)

limestone_stock = excess_limestone.cumsum(axis=1)
consumed_limestone_stock = np.where((limestone_stock > 0) & (limestone_ > 0), np.where(limestone_ <= limestone_stock, limestone_, limestone_stock), 0)
limestone_stock = limestone_stock - consumed_limestone_stock
limestone_extraction = np.where(limestone_ >= consumed_limestone_stock, limestone_ - consumed_limestone_stock, 0)

In [40]:
np.savez_compressed(out_dir / 'aggregate_virgin.npz', data=aggregate_virgin.astype(np.float32))
np.savez_compressed(out_dir / 'aggregate_recycled.npz', data=aggregate_recycled.astype(np.float32))

np.savez_compressed(out_dir / 'excess_sand.npz', data=excess_sand.astype(np.float32))
np.savez_compressed(out_dir / 'excess_gravel.npz', data=excess_gravel.astype(np.float32))

np.savez_compressed(out_dir / 'sand_stock.npz', data=sand_stock.astype(np.float32))
np.savez_compressed(out_dir / 'gravel_stock.npz', data=gravel_stock.astype(np.float32))

np.savez_compressed(out_dir / 'sand_production.npz', data=sand_production.astype(np.float32))
np.savez_compressed(out_dir / 'gravel_production.npz', data=gravel_production.astype(np.float32))

np.savez_compressed(out_dir / 'sand_extraction.npz', data=sand_extraction.astype(np.float32))
np.savez_compressed(out_dir / 'gravel_extraction.npz', data=gravel_extraction.astype(np.float32))

np.savez_compressed(out_dir / 'rec_rate.npz', data=rec_rate.astype(np.float32))
np.savez_compressed(out_dir / 'down_rate.npz', data=down_rate.astype(np.float32))

np.savez_compressed(out_dir / 'EoLWaste.npz', data=EoLWaste.astype(np.float32))
np.savez_compressed(out_dir / 'EoLWaste_recycled.npz', data=EoLWaste_recycled.astype(np.float32))
np.savez_compressed(out_dir / 'EoLWaste_downcycled.npz', data=EoLWaste_downcycled.astype(np.float32))
np.savez_compressed(out_dir / 'EoLWaste_disposed.npz', data=EoLWaste_disposed.astype(np.float32))


In [41]:
np.savez_compressed(out_dir / 'SCM_gypsum.npz', data=SCM_gypsum.astype(np.float32))
np.savez_compressed(out_dir / 'SCM_limestone.npz', data=SCM_limestone.astype(np.float32))
np.savez_compressed(out_dir / 'SCM_pozzolana.npz', data=SCM_pozzolana.astype(np.float32))
np.savez_compressed(out_dir / 'SCM_slag.npz', data=SCM_slag.astype(np.float32))
np.savez_compressed(out_dir / 'SCM_flyash.npz', data=SCM_flyash.astype(np.float32))
np.savez_compressed(out_dir / 'SCM_others.npz', data=SCM_others.astype(np.float32))

In [42]:
np.savez_compressed(out_dir / 'clinker_consumption.npz', data=clinker_consumption.astype(np.float32))
np.savez_compressed(out_dir / 'clinker_production.npz', data=clinker_production.astype(np.float32))
np.savez_compressed(out_dir / 'excess_clinker.npz', data=excess_clinker.astype(np.float32))
np.savez_compressed(out_dir / 'clinker_stock.npz', data=clinker_stock.astype(np.float32))

np.savez_compressed(out_dir / 'CKD.npz', data=CKD.astype(np.float32))

np.savez_compressed(out_dir / 'RawMat_Ca.npz', data=RawMat_Ca.astype(np.float32))
np.savez_compressed(out_dir / 'RawMat_Al.npz', data=RawMat_Al.astype(np.float32))
np.savez_compressed(out_dir / 'RawMat_Fe.npz', data=RawMat_Fe.astype(np.float32))
np.savez_compressed(out_dir / 'RawMat_Si.npz', data=RawMat_Si.astype(np.float32))


np.savez_compressed(out_dir / 'excess_limestone.npz', data=excess_limestone.astype(np.float32))
np.savez_compressed(out_dir / 'limestone_stock.npz', data=limestone_stock.astype(np.float32))
np.savez_compressed(out_dir / 'limestone_extraction.npz', data=limestone_extraction.astype(np.float32))